<a href="https://colab.research.google.com/github/bhavishyapatel004/Data-scientist-project/blob/main/Rice_Leaf_Disease_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rice Leaf Disease Detection Using CNN



In [ ]:
%pip -q install tensorflow numpy pandas matplotlib seaborn scikit-learn pillow


## 1. Imports and Google Drive

In [ ]:
import os, re, hashlib, time, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_recall_fscore_support)
from google.colab import drive

drive.mount('/content/drive')
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 16
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


## 2. Dataset path resolution and recursive image indexing
The indexer searches recursively, assigns each image using its disease-folder ancestor, and standardizes folder labels. If automatic detection selects the wrong location, set `DATASET_PATH` manually.

In [ ]:
TARGETS = {
    'leaf smut': 'Leaf smut',
    'brown spot': 'Brown spot',
    'bacterial leaf blight': 'Bacterial leaf blight'
}

def clean_class_name(folder_name):
    name = re.sub(r'[-_ ]?\d{6,}.*$', '', folder_name).strip().lower()
    for key, display in TARGETS.items():
        if key in name:
            return display
    return None

# Find a likely parent directory containing the three class folders.
DATASET_PATH = None
search_root = '/content/drive/MyDrive'
for root, dirs, files in os.walk(search_root):
    found = {clean_class_name(d) for d in dirs}
    if set(TARGETS.values()).issubset(found):
        DATASET_PATH = root
        break

# If automatic detection does not work, uncomment and edit this line:
# DATASET_PATH = '/content/drive/MyDrive/path/to/dataset'

if DATASET_PATH is None:
    raise FileNotFoundError("Dataset folder not found. Set DATASET_PATH manually.")
print("Dataset root:", DATASET_PATH)

records, unreadable = [], []
for root, _, files in os.walk(DATASET_PATH):
    for file in files:
        if Path(file).suffix.lower() not in VALID_EXTS:
            continue
        path = os.path.join(root, file)
        relative_parts = os.path.relpath(path, DATASET_PATH).split(os.sep)[:-1]
        label = next((clean_class_name(part) for part in reversed(relative_parts)
                      if clean_class_name(part) is not None), None)
        if label is None:
            continue
        try:
            with Image.open(path) as im:
                im.verify()
            with Image.open(path) as im:
                width, height = im.size
            records.append({'filepath': path, 'label': label,
                            'width': width, 'height': height,
                            'aspect_ratio': width / height})
        except Exception as exc:
            unreadable.append({'filepath': path, 'error': str(exc)})

df = pd.DataFrame(records)
if df.empty:
    raise ValueError("No readable images found. Check DATASET_PATH and class folder names.")
EXPECTED_CLASSES = list(TARGETS.values())
df = df[df['label'].isin(EXPECTED_CLASSES)].reset_index(drop=True)
print("Readable image count:", len(df))
print("Class labels:", sorted(df['label'].unique()))
print("Unreadable images:", len(unreadable))
if unreadable:
    display(pd.DataFrame(unreadable).head(10))


## 3. EDA: class distribution, image metadata, duplicates
The outputs below provide quantitative evidence. Review the printed observations before deciding whether imbalance or unusual image geometry needs additional handling.

In [ ]:
class_counts = df['label'].value_counts().reindex(EXPECTED_CLASSES, fill_value=0)
display(class_counts.rename('image_count').to_frame())
print(f"Total readable images: {len(df)}")
print(f"Smallest class: {class_counts.idxmin()} ({class_counts.min()} images)")
print(f"Largest class: {class_counts.idxmax()} ({class_counts.max()} images)")
imbalance_ratio = class_counts.max() / max(class_counts.min(), 1)
print(f"Largest/smallest class ratio: {imbalance_ratio:.2f}")
if imbalance_ratio > 1.5:
    print("Observation: noticeable class imbalance; use training-only class weights.")
else:
    print("Observation: class counts are relatively balanced; still monitor per-class recall.")

plt.figure(figsize=(8,4))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title('Images per disease class'); plt.xlabel('Class'); plt.ylabel('Image count')
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15,4))
sns.histplot(data=df, x='width', hue='label', element='step', ax=axes[0], bins=15)
sns.histplot(data=df, x='height', hue='label', element='step', ax=axes[1], bins=15)
sns.histplot(data=df, x='aspect_ratio', hue='label', element='step', ax=axes[2], bins=15)
axes[0].set_title('Image width'); axes[1].set_title('Image height'); axes[2].set_title('Aspect ratio')
plt.tight_layout(); plt.show()
display(df.groupby('label')[['width','height','aspect_ratio']].describe().round(2))


In [ ]:
# Representative images from each class
fig, axes = plt.subplots(len(EXPECTED_CLASSES), 4, figsize=(12, 9), squeeze=False)
for r, label in enumerate(EXPECTED_CLASSES):
    samples = df[df.label == label].sample(min(4, (df.label == label).sum()), random_state=SEED)
    for c, (_, row) in enumerate(samples.iterrows()):
        with Image.open(row.filepath) as im:
            axes[r,c].imshow(im.convert('RGB'))
        axes[r,c].set_title(label if c == 0 else '')
        axes[r,c].axis('off')
    for c in range(len(samples), 4):
        axes[r,c].axis('off')
plt.suptitle('Representative rice leaf images'); plt.tight_layout(); plt.show()


In [ ]:
# Exact duplicate detection using SHA-256. Keep one copy of exact duplicates.
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

df['sha256'] = df['filepath'].map(sha256_file)
duplicate_rows = df[df.duplicated('sha256', keep=False)].sort_values('sha256')
print("Exact duplicate files:", len(duplicate_rows))
if len(duplicate_rows):
    display(duplicate_rows[['filepath','label','sha256']].head(20))
    # Remove duplicate file copies before splitting to reduce leakage risk.
    df = df.drop_duplicates('sha256', keep='first').drop(columns='sha256').reset_index(drop=True)
else:
    df = df.drop(columns='sha256')
print("Images retained after exact-deduplication:", len(df))
print("Near-duplicate review: exact hashing does not detect resized/recompressed copies; inspect visually if filenames or samples suggest duplicates.")


## 4. Stratified train/validation/test split
Use 70% training, 15% validation, and 15% independent test data. The test set is not used for model selection or tuning. With a small dataset, each split will contain few images; results will therefore have high uncertainty.

In [ ]:
# Confirm that every class has enough examples for a stratified three-way split.
counts = df['label'].value_counts()
if len(counts) != 3 or counts.min() < 3:
    raise ValueError(f"Need at least 3 images in each of the 3 classes for stratified splitting. Counts: {counts.to_dict()}")

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for split_name, split_df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    print(f"\n{split_name}: {len(split_df)} images")
    print(split_df['label'].value_counts().reindex(EXPECTED_CLASSES, fill_value=0))


## 5. Preprocessing and generators
Augmentation is applied only to training data. Validation and test pipelines are deterministic. MobileNetV2 and ResNet50 use their own ImageNet preprocessing functions; custom CNNs use rescaling to [0, 1].

In [ ]:
def make_generators(preprocess_fn=None, augment=False):
    if preprocess_fn is None:
        preprocess_fn = lambda x: x / 255.0
    train_aug = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        rotation_range=20 if augment else 0,
        width_shift_range=0.10 if augment else 0.0,
        height_shift_range=0.10 if augment else 0.0,
        zoom_range=0.10 if augment else 0.0,
        horizontal_flip=augment,
        fill_mode='nearest'
    )
    eval_aug = ImageDataGenerator(preprocessing_function=preprocess_fn)

    common = dict(x_col='filepath', y_col='label', classes=EXPECTED_CLASSES,
                  target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
                  class_mode='categorical', validate_filenames=True)

    train_gen = train_aug.flow_from_dataframe(
        train_df, shuffle=True, seed=SEED, **common)
    val_gen = eval_aug.flow_from_dataframe(
        val_df, shuffle=False, **common)
    test_gen = eval_aug.flow_from_dataframe(
        test_df, shuffle=False, **common)
    return train_gen, val_gen, test_gen

# All class weights are calculated from training labels only.
class_weight_values = compute_class_weight(
    class_weight='balanced',
    classes=np.array(EXPECTED_CLASSES),
    y=train_df['label'].values
)
CLASS_WEIGHTS = {i: float(w) for i, w in enumerate(class_weight_values)}
print("Training class weights:", CLASS_WEIGHTS)

train_base, val_base, test_base = make_generators(augment=False)
train_aug, val_aug, test_aug = make_generators(augment=True)
train_mobile, val_mobile, test_mobile = make_generators(
    tf.keras.applications.mobilenet_v2.preprocess_input, augment=True)
train_resnet, val_resnet, test_resnet = make_generators(
    tf.keras.applications.resnet50.preprocess_input, augment=True)

def make_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
    ]


## 6. Visualize training augmentation

In [ ]:
sample_path = train_df.iloc[0]['filepath']
img = tf.keras.utils.load_img(sample_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
arr = tf.keras.utils.img_to_array(img)
preview_gen = ImageDataGenerator(rotation_range=20, width_shift_range=0.1,
                                 height_shift_range=0.1, zoom_range=0.1,
                                 horizontal_flip=True)
fig, axes = plt.subplots(1, 5, figsize=(14,3))
for i, batch in enumerate(preview_gen.flow(arr[None, ...], batch_size=1, seed=SEED)):
    axes[i].imshow(np.clip(batch[0] / 255.0, 0, 1))
    axes[i].axis('off'); axes[i].set_title(f'Augmented {i+1}')
    if i == 4: break
plt.tight_layout(); plt.show()


## 7. Model 1: baseline custom CNN

In [ ]:
def build_custom_cnn(dropout_rate=0.4, learning_rate=1e-3):
    model = Sequential([
        Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        Conv2D(32, 3, activation='relu'), MaxPooling2D(2),
        Conv2D(64, 3, activation='relu'), MaxPooling2D(2),
        Conv2D(128, 3, activation='relu'), MaxPooling2D(2),
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'), Dropout(dropout_rate),
        Dense(len(EXPECTED_CLASSES), activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model_base_cnn = build_custom_cnn()
history_base_cnn = model_base_cnn.fit(
    train_base, validation_data=val_base, epochs=40,
    callbacks=make_callbacks(), class_weight=CLASS_WEIGHTS, verbose=1)


## 8. Model 2: augmented custom CNN

In [ ]:
model_aug_cnn = build_custom_cnn()
history_aug_cnn = model_aug_cnn.fit(
    train_aug, validation_data=val_aug, epochs=40,
    callbacks=make_callbacks(), class_weight=CLASS_WEIGHTS, verbose=1)


## 9. Model 3: MobileNetV2 transfer learning
Train the classification head with the ImageNet backbone frozen, then optionally fine-tune the final layers using a low learning rate.

In [ ]:
base_mobilenet = MobileNetV2(weights='imagenet', include_top=False,
                             input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
base_mobilenet.trainable = False
model_mobilenet = Sequential([
    Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)), base_mobilenet,
    GlobalAveragePooling2D(), Dense(128, activation='relu'), Dropout(0.4),
    Dense(len(EXPECTED_CLASSES), activation='softmax')
])
model_mobilenet.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                        loss='categorical_crossentropy', metrics=['accuracy'])
history_mobilenet = model_mobilenet.fit(
    train_mobile, validation_data=val_mobile, epochs=30,
    callbacks=make_callbacks(), class_weight=CLASS_WEIGHTS, verbose=1)

# Fine-tune only the last 20 backbone layers; keep BatchNorm layers frozen.
base_mobilenet.trainable = True
for layer in base_mobilenet.layers[:-20]:
    layer.trainable = False
for layer in base_mobilenet.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model_mobilenet.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                        loss='categorical_crossentropy', metrics=['accuracy'])
history_mobilenet_finetune = model_mobilenet.fit(
    train_mobile, validation_data=val_mobile, epochs=12,
    callbacks=make_callbacks(), class_weight=CLASS_WEIGHTS, verbose=1)


## 10. Model 4: ResNet50 transfer learning

In [ ]:
base_resnet = ResNet50(weights='imagenet', include_top=False,
                       input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
base_resnet.trainable = False
model_resnet = Sequential([
    Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)), base_resnet,
    GlobalAveragePooling2D(), Dense(128, activation='relu'), Dropout(0.4),
    Dense(len(EXPECTED_CLASSES), activation='softmax')
])
model_resnet.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                     loss='categorical_crossentropy', metrics=['accuracy'])
history_resnet = model_resnet.fit(
    train_resnet, validation_data=val_resnet, epochs=30,
    callbacks=make_callbacks(), class_weight=CLASS_WEIGHTS, verbose=1)


## 11. Training curves

In [ ]:
def plot_history(histories, names):
    fig, axes = plt.subplots(1, 2, figsize=(14,5))
    for hist, name in zip(histories, names):
        axes[0].plot(hist.history.get('val_accuracy', []), label=name)
        axes[1].plot(hist.history.get('val_loss', []), label=name)
    axes[0].set_title('Validation accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend()
    axes[1].set_title('Validation loss'); axes[1].set_xlabel('Epoch'); axes[1].legend()
    for ax in axes: ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot_history([history_base_cnn, history_aug_cnn, history_mobilenet,
              history_mobilenet_finetune, history_resnet],
             ['Baseline CNN','Augmented CNN','MobileNetV2 frozen',
              'MobileNetV2 fine-tuned','ResNet50 frozen'])


## 12. Validation-based model selection
Choose the candidate using validation macro F1 only. Do not inspect test performance to choose a model.

In [ ]:
models = [model_base_cnn, model_aug_cnn, model_mobilenet, model_resnet]
model_names = ['Baseline CNN', 'Augmented CNN', 'MobileNetV2 fine-tuned', 'ResNet50']
val_generators = [val_base, val_aug, val_mobile, val_resnet]
test_generators = [test_base, test_aug, test_mobile, test_resnet]

def predict_and_metrics(model, generator):
    generator.reset()
    probs = model.predict(generator, verbose=0)
    y_true = generator.classes.copy()
    y_pred = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0)
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision (Macro)': p, 'Recall (Macro)': r, 'F1-Score (Macro)': f1,
        'y_true': y_true, 'y_pred': y_pred, 'probabilities': probs
    }

validation_results = []
for model, name, gen in zip(models, model_names, val_generators):
    m = predict_and_metrics(model, gen)
    validation_results.append({'Model': name, **{k: m[k] for k in
        ['Accuracy','Precision (Macro)','Recall (Macro)','F1-Score (Macro)']}})
df_validation = pd.DataFrame(validation_results).sort_values(
    'F1-Score (Macro)', ascending=False).reset_index(drop=True)
display(df_validation.round(4))
best_model_name = df_validation.iloc[0]['Model']
best_idx = model_names.index(best_model_name)
best_model = models[best_idx]
print("Selected using validation macro F1:", best_model_name)


## 13. Final independent test evaluation
This is the only section that evaluates the selected model on the held-out test set. All reported metrics below are computed from its actual predictions.

In [ ]:
test_gen = test_generators[best_idx]
test_metrics = predict_and_metrics(best_model, test_gen)
print("Selected model:", best_model_name)
print(f"Test accuracy: {test_metrics['Accuracy']:.4f}")
print(f"Test macro precision: {test_metrics['Precision (Macro)']:.4f}")
print(f"Test macro recall: {test_metrics['Recall (Macro)']:.4f}")
print(f"Test macro F1: {test_metrics['F1-Score (Macro)']:.4f}")

class_labels = EXPECTED_CLASSES
print("\nTest classification report:")
print(classification_report(test_metrics['y_true'], test_metrics['y_pred'],
                            labels=range(len(class_labels)),
                            target_names=class_labels, zero_division=0))
cm = confusion_matrix(test_metrics['y_true'], test_metrics['y_pred'],
                      labels=range(len(class_labels)))
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title(f'Independent Test Confusion Matrix: {best_model_name}')
plt.xlabel('Predicted label'); plt.ylabel('True label')
plt.tight_layout(); plt.show()


## 14. Consistent model comparison table
Every model uses the same held-out test images and macro averaging. This table is generated from predictions; no metric values are manually entered. Use it descriptively because the test set is very small.

In [ ]:
test_rows = []
for model, name, gen in zip(models, model_names, test_generators):
    m = predict_and_metrics(model, gen)
    test_rows.append({'Model': name, **{k: m[k] for k in
        ['Accuracy','Precision (Macro)','Recall (Macro)','F1-Score (Macro)']}})
df_test_metrics = pd.DataFrame(test_rows).sort_values(
    'F1-Score (Macro)', ascending=False).reset_index(drop=True)
display(df_test_metrics.round(4))
print("Note: The selected model was chosen on validation data; test metrics are final estimates.")


## 15. Optional reproducible hyperparameter experiment
This small, documented experiment compares dropout and learning-rate settings for the custom CNN using the same train/validation split. It is intentionally limited to avoid overfitting the tiny validation set. Do not use test data for tuning.

In [ ]:
tuning_results = []
for dropout_rate in [0.3, 0.5]:
    for learning_rate in [1e-3, 3e-4]:
        candidate = build_custom_cnn(dropout_rate=dropout_rate,
                                     learning_rate=learning_rate)
        hist = candidate.fit(train_aug, validation_data=val_aug, epochs=20,
                             callbacks=make_callbacks(), class_weight=CLASS_WEIGHTS,
                             verbose=0)
        best_val_f1 = max([
            predict_and_metrics(candidate, val_aug)['F1-Score (Macro)']
        ])
        tuning_results.append({'dropout': dropout_rate,
                               'learning_rate': learning_rate,
                               'best_validation_macro_f1': best_val_f1})
display(pd.DataFrame(tuning_results).sort_values(
    'best_validation_macro_f1', ascending=False))
print("Treat this small search as exploratory; repeated stratified cross-validation is preferable.")


## 16. Model size and inference latency (measured, not hardcoded)

In [ ]:
# Save a temporary copy to measure serialized model size.
with tempfile.TemporaryDirectory() as tmpdir:
    temp_model_path = os.path.join(tmpdir, 'candidate.keras')
    best_model.save(temp_model_path)
    model_size_mb = os.path.getsize(temp_model_path) / (1024 ** 2)

# Warm-up and measure single-image inference latency.
sample_batch, _ = next(iter(test_gen))
single_image = sample_batch[:1]
_ = best_model.predict(single_image, verbose=0)  # warm-up
times = []
for _ in range(20):
    start = time.perf_counter()
    _ = best_model.predict(single_image, verbose=0)
    times.append((time.perf_counter() - start) * 1000)
print(f"Serialized model size: {model_size_mb:.2f} MB")
print(f"Mean single-image inference latency: {np.mean(times):.2f} ms")
print("Latency depends on runtime hardware and is not a mobile-device benchmark.")


## 17. Save selected model and preprocessing metadata

In [ ]:
save_dir = '/content/drive/MyDrive/saved_models'
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, 'rice_leaf_disease_best_model.keras')
best_model.save(model_path)

preprocessing_name = {
    'Baseline CNN': 'pixel_rescale_0_to_1',
    'Augmented CNN': 'pixel_rescale_0_to_1',
    'MobileNetV2 fine-tuned': 'tf.keras.applications.mobilenet_v2.preprocess_input',
    'ResNet50': 'tf.keras.applications.resnet50.preprocess_input'
}[best_model_name]
metadata = {
    'selected_model': best_model_name,
    'class_names': EXPECTED_CLASSES,
    'image_size': [IMG_HEIGHT, IMG_WIDTH],
    'preprocessing': preprocessing_name,
    'selection_metric': 'validation_macro_f1'
}
import json
with open(os.path.join(save_dir, 'rice_leaf_model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)
print("Saved model:", model_path)
print("Saved metadata:", os.path.join(save_dir, 'rice_leaf_model_metadata.json'))


## 18. Conclusion and limitations

In [ ]:
print("RICE LEAF DISEASE DETECTION - EVALUATION SUMMARY")
print("=" * 65)
print("Validation-selected model:", best_model_name)
print(f"Independent test accuracy: {test_metrics['Accuracy']:.4f}")
print(f"Independent test macro F1: {test_metrics['F1-Score (Macro)']:.4f}")
print(f"Model size: {model_size_mb:.2f} MB")
print(f"Measured mean latency: {np.mean(times):.2f} ms per image")
print("\nLimitations:")
print("- The dataset is small, so test metrics have high statistical uncertainty.")
print("- A single stratified split may produce unstable results; use repeated stratified CV when more data is available.")
print("- Exact duplicates were removed, but near-duplicate images may remain.")
print("- Folder labels are assumed to be correct; label quality requires expert verification.")
print("- Field conditions, cameras, lighting and geographic variation are not represented adequately.")
print("- These results do not establish production readiness or validated farmer-facing diagnosis.")
print("\nNext steps:")
print("- Collect substantially more expert-labelled images across farms, seasons and devices.")
print("- Perform repeated stratified cross-validation and report confidence intervals.")
print("- Review per-class errors with an agricultural/domain expert.")
print("- Calibrate predicted probabilities and evaluate on external field images before deployment.")
